In [1]:
import google.generativeai as genai
import json

c:\Users\nphal\fsda-genai_agenticai_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
genai.configure(api_key="API key")

In [3]:
# Load patient data
with open(r"C:\Users\nphal\fsda-genai_agenticai_project\Projects\Usecases\Claimsprocessing\patients_data.json", "r") as f:
    patients = json.load(f)

In [4]:
patients

[{'patient_id': 'P001',
  'name': 'John Doe',
  'age': 45,
  'diagnosis': 'Type 2 Diabetes',
  'cpt_codes': ['83036', '82947'],
  'insurance_plan': {'provider': 'Blue Cross',
   'plan_type': 'PPO',
   'covered_cpt': ['83036', '80053', '99213']}},
 {'patient_id': 'P002',
  'name': 'Jane Smith',
  'age': 62,
  'diagnosis': 'Hypertension',
  'cpt_codes': ['93000', '80061'],
  'insurance_plan': {'provider': 'United Health',
   'plan_type': 'HMO',
   'covered_cpt': ['80061', '99214']}},
 {'patient_id': 'P003',
  'name': 'Rahul Mehta',
  'age': 34,
  'diagnosis': 'Asthma',
  'cpt_codes': ['94010', '99213'],
  'insurance_plan': {'provider': 'Aetna',
   'plan_type': 'POS',
   'covered_cpt': ['94010', '94640', '99213']}}]

In [5]:
# Build a prompt for Gemini
def build_prompt(patients):
    return f"""
You are an insurance claim processing assistant. 
For each patient in the following data, analyze their CPT codes and check if they are covered under their insurance plan.
If a CPT code is not covered, mark it as 'Not Covered'.
Summarize patient details with a recommendation (Approve/Reject).

Patient Data (JSON):
{json.dumps(patients, indent=2)}

Return the summary in the following format:
[
  {{
    "patient_id": "...",
    "name": "...",
    "covered_codes": [...],
    "not_covered_codes": [...],
    "recommendation": "Approve/Reject",
    "reason": "..."
  }},
  ...
]
"""

prompt = build_prompt(patients)


In [6]:
prompt

'\nYou are an insurance claim processing assistant. \nFor each patient in the following data, analyze their CPT codes and check if they are covered under their insurance plan.\nIf a CPT code is not covered, mark it as \'Not Covered\'.\nSummarize patient details with a recommendation (Approve/Reject).\n\nPatient Data (JSON):\n[\n  {\n    "patient_id": "P001",\n    "name": "John Doe",\n    "age": 45,\n    "diagnosis": "Type 2 Diabetes",\n    "cpt_codes": [\n      "83036",\n      "82947"\n    ],\n    "insurance_plan": {\n      "provider": "Blue Cross",\n      "plan_type": "PPO",\n      "covered_cpt": [\n        "83036",\n        "80053",\n        "99213"\n      ]\n    }\n  },\n  {\n    "patient_id": "P002",\n    "name": "Jane Smith",\n    "age": 62,\n    "diagnosis": "Hypertension",\n    "cpt_codes": [\n      "93000",\n      "80061"\n    ],\n    "insurance_plan": {\n      "provider": "United Health",\n      "plan_type": "HMO",\n      "covered_cpt": [\n        "80061",\n        "99214"\n  

In [8]:
model = genai.GenerativeModel("gemini-2.0-flash")  # or latest available

In [9]:
model

genai.GenerativeModel(
    model_name='models/gemini-2.0-flash',
    generation_config={},
    safety_settings={},
    tools=None,
    system_instruction=None,
    cached_content=None
)

In [15]:
response = model.generate_content(prompt)

In [16]:
response.text

"Here's an analysis of the insurance claim dataset based on the provided rules:\n\n**Analysis:**\n\nWe need to check each patient's claim and determine if all their CPT codes are covered under their insurance plan.\n\n*   **Patient P001 (John Doe):**\n    *   CPT Codes: `83036`, `82947`\n    *   Covered CPT Codes: `83036`, `80053`, `99213`\n    *   Analysis: `83036` is covered, but `82947` is NOT covered.\n    *   Recommendation: **Reject**\n\n*   **Patient P002 (Jane Smith):**\n    *   CPT Codes: `93000`, `80061`\n    *   Covered CPT Codes: `80061`, `99214`\n    *   Analysis: `80061` is covered, but `93000` is NOT covered.\n    *   Recommendation: **Reject**\n\n*   **Patient P003 (Rahul Mehta):**\n    *   CPT Codes: `94010`, `99213`\n    *   Covered CPT Codes: `94010`, `94640`, `99213`\n    *   Analysis: `94010` and `99213` ARE both covered.\n    *   Recommendation: **Approve**\n\n**Summary of Findings:**\n\n| Patient ID | Name        | Recommendation | Reason                         

In [20]:
prompt = f"""
Analyze this insurance claim dataset.

Rules:
1. A CPT code must be in 'covered_cpt' list to be approved.
2. If all CPTs are covered → Recommendation = Approve
3. If any CPT is not covered → Recommendation = Reject
4. Summarize the findings clearly.

Data:
{json.dumps(patients, indent=2)}

Return the summary in the following format:

1.About the patient
2. The CPT codes which patient has
3. The CPT codes accepted by the insurance plan
4. Recommedation -- Approved / Rejected

"""

In [37]:
response = model.generate_content(prompt)

In [38]:
response.text

'Here\'s the analysis of the insurance claim dataset, following the specified format:\n\n**Patient P001: John Doe**\n\n1.  **About the patient:** 45-year-old patient diagnosed with Type 2 Diabetes.\n2.  **The CPT codes which patient has:**  "83036", "82947"\n3.  **The CPT codes accepted by the insurance plan:** "83036", "80053", "99213"\n4.  **Recommendation:** Rejected. CPT code "82947" is not covered by the insurance plan.\n\n**Patient P002: Jane Smith**\n\n1.  **About the patient:** 62-year-old patient diagnosed with Hypertension.\n2.  **The CPT codes which patient has:** "93000", "80061"\n3.  **The CPT codes accepted by the insurance plan:** "80061", "99214"\n4.  **Recommendation:** Rejected. CPT code "93000" is not covered by the insurance plan.\n\n**Patient P003: Rahul Mehta**\n\n1.  **About the patient:** 34-year-old patient diagnosed with Asthma.\n2.  **The CPT codes which patient has:** "94010", "99213"\n3.  **The CPT codes accepted by the insurance plan:** "94010", "94640", "

In [41]:
import pandas as pd
import ast
import re

raw_output = response.text.strip()
results = None

try:
    # --- Step 1: Extract JSON-like list ---
    match = re.search(r'(\[.*\])', raw_output, re.DOTALL)
    if match:
        cleaned_text = match.group(1)
    else:
        print("No JSON block found in Gemini output.")
        print("Raw output preview:", raw_output[:3000])
        cleaned_text = None

    # --- Step 2: Parse if found ---
    if cleaned_text:
        try:
            results = json.loads(cleaned_text)
        except json.JSONDecodeError:
            results = ast.literal_eval(cleaned_text)

    # --- Step 3: Show clean result summary ---
    if results:
        print("Successfully parsed Gemini output.")
    else:
        print("Gemini output was not valid JSON; please check model response.")

except Exception as e:
    # Clean catch without traceback
    print(f"Error while parsing Gemini output: {str(e)}")


No JSON block found in Gemini output.
Raw output preview: Here's the analysis of the insurance claim dataset, following the specified format:

**Patient P001: John Doe**

1.  **About the patient:** 45-year-old patient diagnosed with Type 2 Diabetes.
2.  **The CPT codes which patient has:**  "83036", "82947"
3.  **The CPT codes accepted by the insurance plan:** "83036", "80053", "99213"
4.  **Recommendation:** Rejected. CPT code "82947" is not covered by the insurance plan.

**Patient P002: Jane Smith**

1.  **About the patient:** 62-year-old patient diagnosed with Hypertension.
2.  **The CPT codes which patient has:** "93000", "80061"
3.  **The CPT codes accepted by the insurance plan:** "80061", "99214"
4.  **Recommendation:** Rejected. CPT code "93000" is not covered by the insurance plan.

**Patient P003: Rahul Mehta**

1.  **About the patient:** 34-year-old patient diagnosed with Asthma.
2.  **The CPT codes which patient has:** "94010", "99213"
3.  **The CPT codes accepted by the i